# Sinhala Document OCR - Colab Baseline Pipeline

This notebook runs an **end-to-end baseline** for the Sinhala line-recognition
CRNN (CNN + BiLSTM + CTC) on a Colab **GPU** runtime. Each step maps directly onto
the project's `src/` modules so you can read the code alongside the notebook.

**Pipeline**
1. Clone the repository and install dependencies.
2. Install a Sinhala-capable font (Colab is Linux, so Windows fonts are absent).
3. Generate a synthetic Sinhala line dataset with a seeded train/val/test split.
4. Train the CRNN for a few epochs, logging train loss and validation CER, saving the best checkpoint.
5. Evaluate on the held-out test set (mean CER / WER) with qualitative examples.
6. Run an inference demo on a single line image.

> **Tip:** enable the GPU via *Runtime -> Change runtime type -> Hardware accelerator: GPU*.

## 1. Clone the project and install dependencies

We clone the public repository, `cd` into it, and install the Python requirements.
The notebook is **OS-aware**: it detects whether it is running inside Colab (Linux).

In [ ]:
import os

REPO_URL = "https://github.com/Hellsgate96/sinhala-document-ocr-2026.git"
REPO_DIR = "sinhala-document-ocr-2026"
REPO_BRANCH = "master"

# Optional GitHub token (set the GITHUB_TOKEN env var) for private-repo access.
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
AUTH_URL = (
    REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    if GITHUB_TOKEN else REPO_URL
)

# Detect Colab (Linux) vs. a local run.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        # Clone the correct branch explicitly to avoid default-branch confusion.
        !git clone -b {REPO_BRANCH} {AUTH_URL} {REPO_DIR}
    else:
        # Repo already present: update it from the correct branch.
        !git -C {REPO_DIR} pull origin {REPO_BRANCH}

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
print("Working directory:", os.getcwd())

In [ ]:
# torch / numpy / Pillow are usually preinstalled in Colab; this is quick if so.
!pip -q install -r requirements.txt
# Optional: faster CER/WER (editdistance is not required)
!pip -q install -r requirements-optional.txt || echo "Optional metrics deps skipped"



## 2. Install a Sinhala-capable font

The synthetic generator renders text with a TrueType font. Colab runs on Linux, so
the Windows default (`Nirmala.ttc`) does not exist. We install **Noto Sans Sinhala**
from the `fonts-noto-core` apt package, with a direct download as a robust fallback,
then locate the resulting `.ttf` on disk.

In [ ]:
import glob

FONT_PATH = None
if IN_COLAB:
    # 1) Noto Sans Sinhala ships inside the fonts-noto-core apt package.
    !apt-get -qq update > /dev/null 2>&1
    !apt-get -qq install -y fonts-noto-core > /dev/null 2>&1 || true
    # 2) Robust fallback: download a static Noto Sans Sinhala TTF.
    os.makedirs("fonts", exist_ok=True)
    if not glob.glob("fonts/*Sinhala*.ttf"):
        !wget -q -O fonts/NotoSansSinhala-Regular.ttf "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansSinhala/NotoSansSinhala-Regular.ttf" || true
    # v3: a few extra Sinhala font families for training diversity (best-effort; single
    # font is still fine if these fail - see the RuntimeError guard below).
    if not os.path.isfile("fonts/NotoSerifSinhala-Regular.ttf"):
        !wget -q -O fonts/NotoSerifSinhala-Regular.ttf "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSerifSinhala/NotoSerifSinhala-Regular.ttf" || true
    if not os.path.isfile("fonts/AbhayaLibre-Regular.ttf"):
        !wget -q -O fonts/AbhayaLibre-Regular.ttf "https://github.com/google/fonts/raw/main/ofl/abhayalibre/AbhayaLibre-Regular.ttf" || true
    if not os.path.isfile("fonts/Yaldevi-Variable.ttf"):
        !wget -q -O fonts/Yaldevi-Variable.ttf "https://github.com/google/fonts/raw/main/ofl/yaldevi/Yaldevi%5Bwght%5D.ttf" || true

# Search common locations for a Sinhala-capable font file.
candidates = (
    glob.glob("/usr/share/fonts/**/*Sinhala*.ttf", recursive=True)
    + glob.glob("fonts/*Sinhala*.ttf")
    + ["C:/Windows/Fonts/Nirmala.ttc"]  # local Windows fallback
)
candidates = [c for c in candidates if os.path.isfile(c)]
assert candidates, "No Sinhala font found - re-run this cell or `apt-get install fonts-noto-core`."
FONT_PATH = candidates[0]
# v3: use every font family we found (diversity), not just the first - training on a
# single font family was a real cause of poor real-world generalisation (see README).
extra_fonts = [c for c in glob.glob("fonts/*.ttf") if os.path.isfile(c)]
FONT_PATHS_ALL = sorted(set(candidates + extra_fonts))
print("Using Sinhala font (primary):", FONT_PATH)
print("All font faces available for training:", FONT_PATHS_ALL)

## 3. Load the configuration

`configs/default.yaml` is the single source of truth for paths, image geometry,
synthetic-data settings and training hyper-parameters. We load it, point the
generator at the Colab font, and choose baseline dataset / training sizes.

In [ ]:
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.utils.common import load_config, get_logger
from src.utils.display import configure_display_utf8, setup_matplotlib_sinhala

configure_display_utf8()

cfg = load_config("configs/default.yaml")

# OS-aware: use every font family we found (v3: font diversity, see README).
cfg["synthetic"]["fonts"] = FONT_PATHS_ALL
MPL_FONT = setup_matplotlib_sinhala(FONT_PATH)
print("Matplotlib Sinhala font:", MPL_FONT or "(not found)")

# Taller lines help Sinhala conjuncts (must match checkpoint after training).
cfg["image"]["height"] = 48
cfg["inference"]["image_height"] = 48

# Baseline sizes - tune these for longer runs.
NUM_SAMPLES = 15000     # document OCR: 15k+ synthetic lines
EPOCHS = 25
BATCH_SIZE = 64
OUT_DIR = cfg["paths"]["synthetic_dir"]   # data/synthetic

print("image height :", cfg["image"]["height"], "(train + inference must match checkpoint)")
print("max_width    :", cfg["image"]["max_width"])
print("charset path :", cfg["paths"]["charset_path"])
print("output dir   :", OUT_DIR)


## 4. Generate the synthetic dataset

`generate(...)` renders `NUM_SAMPLES` augmented line images, then writes
`all_labels.txt` plus the per-split `train_labels.txt` / `val_labels.txt` /
`test_labels.txt` (each row is `relative_image_path<TAB>transcript`). Vocabulary is
drawn from `sample_words.txt` and the form-field list `form_vocab.txt`. A `tqdm`
progress bar shows rendering progress.

In [ ]:
from src.data.synthetic_generator import generate, load_corpus, load_word_lists

logger = get_logger("generate")
words = load_word_lists([cfg["paths"]["word_list"], cfg["paths"].get("form_vocab")],
                        warn=logger.warning)
print("Vocabulary entries:", len(words))
# v3 fix: this cell previously never loaded the real-sentence corpus, so Colab runs
# trained on much less diverse text than the local pipeline - load and pass it here too.
corpus = load_corpus(cfg["paths"].get("corpus"), warn=logger.warning)
print("Corpus lines:", len(corpus))

counts = generate(
    out_dir=OUT_DIR,
    num_samples=NUM_SAMPLES,
    font_paths=cfg["synthetic"]["fonts"],
    font_sizes=cfg["synthetic"]["font_sizes"],
    words=words,
    min_words=cfg["synthetic"]["min_words"],
    max_words=cfg["synthetic"]["max_words"],
    augment=cfg["synthetic"]["augment"],
    split=cfg["synthetic"]["split"],
    seed=cfg["project"]["seed"],
    logger=logger,
    numeric_ratio=cfg["synthetic"]["numeric_ratio"],
    mixed_ratio=cfg["synthetic"]["mixed_ratio"],
    corpus=corpus,
    corpus_ratio=float(cfg["synthetic"].get("corpus_ratio", 0.65)),
)
print("Split counts:", counts)

## 4b. Optional: detector-in-the-loop page supplement (v3 domain-gap fix)

Line-crop synthetic CER alone is **not** a reliable signal for real-world performance (see README "v3 domain-gap fix"). This renders full synthetic pages, runs the real `ProjectionLineDetector` over them, and trains on the detector's *actual* output crops instead of only idealised single-line crops. Recommended for any real training run; set `NUM_PAGES = 0` to skip.


In [ ]:
from src.data.page_synth import LAYOUTS, generate_detector_in_the_loop

NUM_PAGES = 3000  # set to 0 to skip this supplement
PAGES_DIR = cfg["paths"].get("synthetic_pages_dir", "data/synthetic_pages")
if NUM_PAGES > 0:
    stats = generate_detector_in_the_loop(
        out_dir=PAGES_DIR,
        num_pages=NUM_PAGES,
        font_paths=cfg["synthetic"]["fonts"],
        font_sizes=cfg["synthetic"]["font_sizes"],
        corpus=corpus,
        detection_cfg=cfg.get("detection", {}),
        layouts=LAYOUTS,
        seed=cfg["project"]["seed"] + 1,
        logger=logger,
    )
    print("Page-supplement crops:", stats["num_crops"], "| per-layout detector match rate:", stats["match_rate"])
else:
    print("Skipped page supplement (NUM_PAGES=0)")


### Preview a few generated lines

`setup_matplotlib_sinhala()` (Section 3) registers Noto Sans Sinhala for plot titles. OCR itself uses PNG pixels and label files; this is for **visual verification** only.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from src.data.dataset import read_labels

rows = read_labels(os.path.join(OUT_DIR, "train_labels.txt"))
fig, axes = plt.subplots(5, 1, figsize=(10, 7))
for ax, (rel, text) in zip(axes, rows[:5]):
    ax.imshow(Image.open(os.path.join(OUT_DIR, rel)))
    ax.set_title(f"Ground truth (train): {text[:60]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Train the CRNN baseline

`src/recognition/train.py` builds the shared `Charset` (saved next to the
checkpoints), trains with `torch.nn.CTCLoss`, logs **train loss** and **validation
CER** each epoch, and saves `crnn_best.pth` (lowest val CER) + `crnn_last.pth` to
`models/`. We pass config overrides on the command line.

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

cmd = (
    f"python -m src.recognition.train --config configs/default.yaml "
    f"paths.synthetic_dir={OUT_DIR} train.epochs={EPOCHS} "
    f"train.batch_size={BATCH_SIZE} train.device={DEVICE}"
)
# v3: merge the detector-in-the-loop page-crop supplement (Section 4b) when present -
# this closes the train/inference distribution gap; see README "v3 domain-gap fix".
PAGES_TRAIN_LABELS = os.path.join(cfg["paths"].get("synthetic_pages_dir", "data/synthetic_pages"), "train_labels.txt")
if os.path.isfile(PAGES_TRAIN_LABELS):
    cmd += f" --extra-labels {PAGES_TRAIN_LABELS}"
POEM_AUG_LABELS = "data/real/labels/poem_kanyawee_aug.txt"
if os.path.isfile(POEM_AUG_LABELS):
    cmd += f" --extra-labels {POEM_AUG_LABELS}"
print(cmd)
!{cmd}


## 6. Evaluate on the held-out test set

We load the best checkpoint and the saved charset, build a test `DataLoader`, and
compute corpus **CER** and **WER** with `evaluate_model(...)`.

In [ ]:
from src.charset import Charset
from src.data.dataset import build_dataloader
from src.recognition.model import build_crnn
from src.evaluation.metrics import evaluate_model
from src.utils.common import load_checkpoint, get_device

charset = Charset.load(cfg["paths"]["charset_path"])
device = get_device(DEVICE)
model = build_crnn(charset.num_classes, cfg.get("model"),
                   in_channels=cfg["image"]["channels"]).to(device)
load_checkpoint("models/crnn_best.pth", model, map_location=str(device))
model.eval()

test_loader = build_dataloader(
    os.path.join(OUT_DIR, "test_labels.txt"), charset,
    batch_size=cfg["train"]["batch_size"], height=cfg["image"]["height"],
    max_width=cfg["image"]["max_width"], channels=cfg["image"]["channels"],
    shuffle=False, num_workers=2)

report = evaluate_model(model, test_loader, charset, device=device, measure_cpu_time=False)
print(f"TEST  samples={report['num_samples']}  CER={report['cer']:.4f}  WER={report['wer']:.4f}")

### Qualitative predictions (image + ground truth + prediction)

In [ ]:
test_rows = read_labels(os.path.join(OUT_DIR, "test_labels.txt"))
n = min(5, len(test_rows))
fig, axes = plt.subplots(n, 1, figsize=(10, 1.8 * n))
if n == 1:
    axes = [axes]
for ax, (rel, _), sample in zip(axes, test_rows[:n], report["per_sample"][:n]):
    ax.imshow(Image.open(os.path.join(OUT_DIR, rel)))
    ax.set_title(
        f"GT: {sample['ref'][:40]}\nPRED: {sample['hyp'][:40]} (CER={sample['cer']:.2f})",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Optional: quick check on a synthetic test line

If you want a one-line sanity check before trying your own photos, run the cell below.
It runs `predict_image(...)` on a single line from the synthetic **test** split (ground
truth is known). For a full document workflow (upload → preprocess → line detection → OCR),
continue to **Section 8**.


In [ ]:
from IPython.display import display
from PIL import Image
from src.recognition.predict import predict_image

sample_rel, sample_gt = read_labels(os.path.join(OUT_DIR, "test_labels.txt"))[0]
sample_path = os.path.join(OUT_DIR, sample_rel)
pred = predict_image(model, charset, sample_path,
                     cfg["image"]["height"], cfg["image"]["max_width"],
                     cfg["image"]["channels"], device)
print("Image       :", sample_path)
print("Ground truth:", sample_gt)
print("Prediction  :", pred)
display(Image.open(sample_path))


## 8. Test on your own uploaded image

Upload a photo or scan of a Sinhala document (form, invoice, ID, handwritten note, etc.).

**Pipeline:** upload -> optional preprocessing -> text-line detection -> CRNN recognition per line -> results.

**Checkpoint:** this section loads `models/crnn_best.pth` and `models/charset.json` produced in **Section 5**.
If you have not trained yet, run training first, or set `CHECKPOINT_PATH` / `CHARSET_PATH` to a copy you saved
(for example from Google Drive).

**Domain gap:** the CRNN is trained mainly on **synthetic plain black text on white paper**. Stylized logos,
outlines, or graphics (e.g. white text on a dark background) may still be wrong until you fine-tune on real data.
Best results: a clear photo of **printed Sinhala on white paper**.

**Tips:** printed pages with clear contrast work best with the OpenCV line detector. If detection finds no lines,
the notebook can treat the whole image as one line (see toggles in the next cell).

**Recognition height:** `RECOGNITION_HEIGHT` comes from `configs/default.yaml` (`inference.image_height`). If you change it from the height used when the checkpoint was trained, you **must retrain** (Section 5) before expecting good uploads.

**Undertrained model:** if validation CER from training was **> 0.5**, increase `NUM_SAMPLES` and/or `EPOCHS` and train again.


In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt

# --- Upload inference toggles (Section 8) ---
INVERT_IF_DARK_BG = True
SKIP_DOCUMENT_BINARIZE = True
USE_WHOLE_IMAGE_IF_ONE_LINE = True

UPLOAD_DIR = os.path.join(cfg["paths"]["data_dir"], "uploads")
os.makedirs(UPLOAD_DIR, exist_ok=True)

UPLOADED_IMAGE_PATH = None

if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file selected. Run the cell again and choose an image.")
    fname = next(iter(uploaded))
    UPLOADED_IMAGE_PATH = os.path.join(UPLOAD_DIR, fname)
    with open(UPLOADED_IMAGE_PATH, "wb") as f:
        f.write(uploaded[fname])
else:
    UPLOADED_IMAGE_PATH = globals().get("LOCAL_UPLOAD_PATH", "").strip()
    if not UPLOADED_IMAGE_PATH:
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"):
            hits = sorted(glob.glob(os.path.join(UPLOAD_DIR, ext)))
            if hits:
                UPLOADED_IMAGE_PATH = hits[-1]
                break
    if not UPLOADED_IMAGE_PATH or not os.path.isfile(UPLOADED_IMAGE_PATH):
        print(
            "Local mode: set LOCAL_UPLOAD_PATH = r'C:\\path\\to\\document.jpg' "
            "in a cell above, or place an image in data/uploads/, then re-run this cell."
        )
    else:
        print("Using local file:", UPLOADED_IMAGE_PATH)

if UPLOADED_IMAGE_PATH and os.path.isfile(UPLOADED_IMAGE_PATH):
    print("Saved to:", UPLOADED_IMAGE_PATH)
    bgr = cv2.imread(UPLOADED_IMAGE_PATH)
    if bgr is None:
        raise ValueError(f"Could not read image: {UPLOADED_IMAGE_PATH}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(rgb)
    plt.title("Uploaded document")
    plt.axis("off")
    plt.show()


In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import torch

from src.charset import Charset
from src.detection.text_detection import OpenCVLineDetector, crop_lines, draw_boxes
from src.preprocessing.preprocess import preprocess_document
from src.recognition.inference import inference_options_from_config, prepared_line_for_display
from src.recognition.model import build_crnn
from src.recognition.predict import predict_line_array
from src.utils.common import configure_stdout_utf8, get_device, load_checkpoint

configure_stdout_utf8()

SHOW_PREPROCESSED = True
RUN_PREPROCESS = True and not SKIP_DOCUMENT_BINARIZE
MIN_LINE_HEIGHT = cfg["detection"]["min_line_height"]
MIN_LINE_WIDTH = cfg["detection"]["min_line_width"]
DILATE_KERNEL = tuple(cfg["detection"]["dilate_kernel"])
INF_OPTS = inference_options_from_config(cfg)
RECOGNITION_HEIGHT = INF_OPTS["height"]
print(f"Recognition input height: {RECOGNITION_HEIGHT}px (must match training)")

CHECKPOINT_PATH = os.path.join(cfg["paths"]["models_dir"], "crnn_best.pth")
CHARSET_PATH = cfg["paths"]["charset_path"]

if not UPLOADED_IMAGE_PATH or not os.path.isfile(UPLOADED_IMAGE_PATH):
    raise RuntimeError("Run the upload cell above first (Section 8).")

if not os.path.isfile(CHECKPOINT_PATH):
    raise FileNotFoundError(
        "Train the model first (Section 5) or place crnn_best.pth in models/"
    )
if not os.path.isfile(CHARSET_PATH):
    raise FileNotFoundError(
        f"Charset file not found: {CHARSET_PATH}. Run training (Section 5) first."
    )

_device = globals().get("device")
if _device is None:
    _device = get_device(globals().get("DEVICE", "auto"))

_charset = Charset.load(CHARSET_PATH)
_model = build_crnn(
    _charset.num_classes,
    cfg.get("model"),
    in_channels=INF_OPTS["channels"],
).to(_device)
_ckpt_meta = load_checkpoint(CHECKPOINT_PATH, _model, map_location=str(_device))
_val_cer = _ckpt_meta.get("cer")
if _val_cer is not None and float(_val_cer) > 0.5:
    print(
        f"[warn] checkpoint val CER={float(_val_cer):.3f} (>0.5) — model undertrained; "
        "increase NUM_SAMPLES / EPOCHS and retrain."
    )
if int(cfg["image"]["height"]) != RECOGNITION_HEIGHT:
    print("[warn] cfg image.height != inference height — keep them aligned.")
_model.eval()

page_bgr = cv2.imread(UPLOADED_IMAGE_PATH, cv2.IMREAD_COLOR)
if page_bgr is None:
    raise ValueError(f"Could not read {UPLOADED_IMAGE_PATH}")

page_gray = cv2.cvtColor(page_bgr, cv2.COLOR_BGR2GRAY)

if RUN_PREPROCESS:
    page_for_det = preprocess_document(page_bgr)
elif SKIP_DOCUMENT_BINARIZE:
    page_for_det = page_gray
else:
    page_for_det = page_gray

h_img, mw, ch = INF_OPTS["height"], INF_OPTS["max_width"], INF_OPTS["channels"]
min_model_w = INF_OPTS.get("min_model_width", 0)
pad_to_h = INF_OPTS.get("pad_to_height", True)
auto_inv = INVERT_IF_DARK_BG and INF_OPTS["auto_invert"]
denoise = INF_OPTS["denoise"]

skip_detection = False
if USE_WHOLE_IMAGE_IF_ONE_LINE:
    ph, pw = page_gray.shape[:2]
    if pw >= 4 * ph or ph <= 3 * max(int(MIN_LINE_HEIGHT), 1):
        skip_detection = True
        print("[info] Single-line crop heuristic: using full image for recognition.")

if skip_detection:
    boxes = [(0, 0, page_gray.shape[1], page_gray.shape[0])]
else:
    detector = OpenCVLineDetector(
        dilate_kernel=DILATE_KERNEL,
        min_line_height=int(MIN_LINE_HEIGHT),
        min_line_width=int(MIN_LINE_WIDTH),
    )
    boxes = detector.detect(page_for_det)
    if not boxes:
        h, w = page_for_det.shape[:2]
        boxes = [(0, 0, w, h)]
        print("[info] No lines detected -- using the full image as one line.")

line_crops = crop_lines(
    page_gray,
    boxes,
    padding_x=int(cfg["detection"].get("crop_padding_x", 10)),
    padding_y=int(cfg["detection"].get("crop_padding_y", 5)),
    min_crop_height=int(cfg["detection"].get("min_crop_height", 14)),
)
annotated_bgr = draw_boxes(page_bgr, boxes)
annotated = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

upload_predictions = []

with torch.no_grad():
    for i, crop in enumerate(line_crops):
        prepared_view = prepared_line_for_display(
            crop,
            height=h_img,
            max_width=mw,
            auto_invert=auto_inv,
            denoise=denoise,
            min_model_width=min_model_w,
            pad_to_height=pad_to_h,
        )
        text = predict_line_array(
            _model,
            _charset,
            crop,
            h_img,
            mw,
            ch,
            _device,
            auto_invert=auto_inv,
            denoise=denoise,
            min_model_width=min_model_w,
            pad_to_height=pad_to_h,
            warn_garbage=True,
        )
        upload_predictions.append(
            {
                "line": i + 1,
                "text": text,
                "crop": crop.copy(),
                "prepared": prepared_view,
            }
        )
        print(f"Line {i + 1} prediction:", text)

full_transcription = "\n".join(p["text"] for p in upload_predictions)
print("--- Full document transcription (copy-friendly) ---")
print(full_transcription)
print("--- end ---")

if SHOW_PREPROCESSED:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(cv2.cvtColor(page_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original upload")
    axes[0].axis("off")
    axes[1].imshow(page_for_det, cmap="gray")
    axes[1].set_title("Page used for detection (grayscale)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

print(f"{'Line':>4}  Predicted text")
print("-" * 40)
for row in upload_predictions:
    print(f"{row['line']:>4}  {row['text']}")

plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.title("Detected text lines (red boxes)")
plt.axis("off")
plt.show()

n = len(upload_predictions)
if n:
    fig, axes = plt.subplots(n, 2, figsize=(12, max(2, 1.8 * n)))
    if n == 1:
        axes = [axes]
    for ax_row, row in zip(axes, upload_predictions):
        ax_row[0].imshow(row["crop"], cmap="gray")
        ax_row[0].set_title(f"Line {row['line']} crop")
        ax_row[0].axis("off")
        ax_row[1].imshow(row["prepared"], cmap="gray")
        ax_row[1].set_title(f"Line {row['line']}: {row['text'][:60]}")
        ax_row[1].axis("off")
    plt.tight_layout()
    plt.show()


## Optional: persist the model to Google Drive

Uncomment to mount Drive and copy the best checkpoint + charset so they survive the
Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
os.makedirs('/content/drive/MyDrive/sinhala_ocr', exist_ok=True)
shutil.copy('models/crnn_best.pth', '/content/drive/MyDrive/sinhala_ocr/')
shutil.copy('models/charset.json', '/content/drive/MyDrive/sinhala_ocr/')
print("Uncomment the lines above to save the model + charset to Google Drive.")